In [29]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

def cuts(df_):
    base = (
        (df_['n_ak4jets']   >= 5)       &
        (df_['n_b_outZH']   == 2)       &
        # ---------------------------------------------------------
        # WARNING: I commented out the old ZH_bbvLscore cut. 
        # Cutting on an old tagger will sculpt the distributions 
        # of the new taggers and ruin your shape comparison!
        # ---------------------------------------------------------
        # (df_['ZH_bbvLscore'] < 0.5669) & 
        (df_['ZH_M']        >= 50)      &
        (df_['MET_pt']      > 20)       &
        (df_['ZH_M']        <= 200)
    )
    return base

# ==========================================
# 1. GLOBAL SETTINGS & CONFIGURATION
# ==========================================
hep.style.use(hep.style.CMS)

COFFEA_DIR = "Taggers/" # <-- UPDATE THIS PATH
LUMI = 110 # in fb^-1 

# --- Processes (NO DATA) ---
bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
sig_processes = ['ttZ', 'ttH']
all_processes = bkg_processes + sig_processes

# --- Variables to Compare ---
tagger_vars = [
    'ZH_GloParT3_Xbb', 
    'ZH_GloParT3_QCD', 
    'ZH_PNet_XbbVsQCD', 
    'ZH_PNetLegacy_XbbVsQCD',
    'ZH_bbvLscore'
]

cut_vars = ['ZH_M', 'n_b_outZH', 'n_ak4jets', 'MET_pt']

# Default to standard tagger binning (0 to 1)
binning_dict = {
    'ZH_GloParT3_Xbb': (40, 0, 1),
    'ZH_GloParT3_QCD': (40, 0, 1),
    'ZH_PNet_XbbVsQCD': (40, 0, 1),
    'ZH_PNetLegacy_XbbVsQCD': (40, 0, 1),
    'ZH_bbvLscore': (40, 0, 1)
}
default_binning = (40, 0, 1)

bkg_colors = {'VJets':'#3f90da', 'QCD':'#ffa90e', 'tt_B':'#bd1f01', 'TTBar':'#94a4a2', 'SingleTop':'#e76300', 'TTX':'#b9ac70'}
sig_colors = {'ttZ':'#832db6', 'ttH':'#a96b59'}
mc_colors = {**bkg_colors, **sig_colors}

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf, t\bar{t}+cc$', 'SingleTop': 'single top',
    'TTX': r'$t\bar{t}t\bar{t}, t\bar{t}W, t\bar{t}HW, t\bar{t}Hq$',
    'ttZ': r'$t\bar{t}Z$', 'ttH': r'$t\bar{t}H$'
}

# ==========================================
# 2. DATA EXTRACTION & WEIGHTING FUNCTIONS
# ==========================================
def extract_nominal_genweights(coffea_dir):
    target_weights = {
        'tttolnu2q': 480547550.0,
        'ttto2l2nu': 466318140.0,
        'ttto4q': 468705400.0,
        'ttbbtolnu2q': 21007254.0,
        'ttbbto2l2nu': 11683236.0,
        'ttbbto4q': 14012432.0
    }
    
    nom_genweights = {}
    nom_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    
    for file_path in nom_files:
        filein = load(file_path)
        gw_dict = filein.get('sum_signOf_genweights', {})
        
        for dataset, weight in gw_dict.items():
            if isinstance(weight, dict):
                for sub_dataset, sub_weight in weight.items():
                    clean_name = sub_dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                    nom_genweights[sub_dataset] = target_weights.get(clean_name, sub_weight)
            else:
                clean_name = dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                nom_genweights[dataset] = target_weights.get(clean_name, weight)
                
    return nom_genweights

def getZhbbWeight(df_, year=None):
    if 'norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) 

    weight = df_['norm_weight'].copy()
    
    # ==========================================
    # THE LUMINOSITY PATCH
    # Rescale from 2017 default to 2024 target
    # ==========================================
    old_lumi = 41.529
    target_lumi = 110.0
    weight *= (target_lumi / old_lumi)
    # ==========================================

    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    
    # Be sure to uncomment 'btag_sf' now that the normalization is fixed!
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf','topptWeight', 'puWeight', 'ele_trig_sf']#, 'btag_sf']  
    
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
        
    return weight

def load_and_cut_data(coffea_dir=COFFEA_DIR, nom_genweights=None):
    tracked_data = {proc: {} for proc in all_processes}
    valid_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    
    print(f"Extracting nominal data from {len(valid_files)} files...")

    vars_to_extract = tagger_vars + ['norm_weight', 'genWeight'] + cut_vars

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            if 'TTTo' in raw_proc:
                if 'tt+B' in raw_proc: continue 
                mapped_proc = 'TTBar'
            elif 'TTbb' in raw_proc:
                if 'tt+B' not in raw_proc: continue 
                mapped_proc = 'tt_B'
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: mapped_proc = 'VJets'
            elif 'QCD' in raw_proc: mapped_proc = 'QCD'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower(): mapped_proc = 'ttZ'
            elif 'TTLL' in raw_proc or 'TTNuNu' in raw_proc: mapped_proc = 'ttZ'
            elif 'SingleTop' in raw_proc: mapped_proc = 'SingleTop'
            elif 'TTX' in raw_proc: mapped_proc = 'TTX'
            
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():
                genweight = nom_genweights.get(dataset, 1.0) if nom_genweights else genweight_dict.get(dataset, 1.0)
                if isinstance(genweight, dict): genweight = genweight.get(dataset, 1.0)
                
                try:
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue 
                    
                tmp_data = {}
                skip_dataset = False
                
                for var in vars_to_extract:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    
                    if dict_key in nom_dict:
                        arr = np.array(nom_dict[dict_key].value)
                        if var == 'norm_weight':
                            tmp_data[var] = arr / genweight
                        else:
                            tmp_data[var] = arr
                    elif var in cut_vars or var == 'norm_weight':
                        skip_dataset = True
                        break 
                        
                if skip_dataset or not tmp_data: continue 
                
                df = pd.DataFrame(tmp_data)
                if dataset not in tracked_data[mapped_proc]:
                    tracked_data[mapped_proc][dataset] = []
                tracked_data[mapped_proc][dataset].append(df)

    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = []
        for dataset, branches in tracked_data[proc].items():
            dfs_to_concat.extend(branches)
            
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            df = df[cuts(df)] 
            df['tot_weight'] = getZhbbWeight(df)
            data_dict[proc] = df
        else:
            data_dict[proc] = None
            
    return data_dict

# ==========================================
# 3. PRE-COMPUTE HISTOGRAMS
# ==========================================
print("\n--- Starting Data Extraction & Histogramming ---")
global_nom_genweights = extract_nominal_genweights(COFFEA_DIR)
current_data = load_and_cut_data(nom_genweights=global_nom_genweights)

hist_dict = {v: {} for v in tagger_vars}

for var in tagger_vars:
    n_bins, x_min, x_max = binning_dict.get(var, default_binning)
    bins = np.linspace(x_min, x_max, n_bins + 1)
    
    for proc in all_processes:
        df = current_data[proc]
        if df is None or df.empty or var not in df.columns:
            hist_dict[var][proc] = {'counts': np.zeros(n_bins)}
            continue
            
        mask = df[var].notna()
        vals = np.clip(df[var][mask], None, bins[-1])
        weights = df['tot_weight'][mask].copy()
        
        if proc == 'tt_B': weights *= 1.318 # k-factor
        
        counts, _ = np.histogram(vals, bins=bins, weights=weights)
        hist_dict[var][proc] = {'counts': counts}

# ==========================================
# 4. PLOTTING FUNCTION (SHAPE COMPARISON)
# ==========================================
def plot_shapes_to_pdf(var_names, hist_dictionary, output_filename="Tagger_Shape_Comparison.pdf"):
    with PdfPages(output_filename) as pdf:
        
        # Create a 2x2 grid for the 4 taggers
        fig, axes = plt.subplots(3, 2, figsize=(20, 18))
        axes = axes.flatten()
        
        for idx, var_name in enumerate(var_names):
            ax = axes[idx]
            
            n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
            bins = np.linspace(x_min, x_max, n_bins + 1)
            
            # 1. Gather & Normalize Backgrounds
            bkg_hists, bkg_labels, bkg_colors_list = [], [], []
            total_bkg_yield = 0
            
            for proc in bkg_processes:
                counts = hist_dictionary[var_name][proc]['counts']
                total_bkg_yield += np.sum(counts)
                
            for proc in bkg_processes:
                counts = hist_dictionary[var_name][proc]['counts']
                # Normalize so the entire background stack integrates to 1
                norm_counts = counts / total_bkg_yield if total_bkg_yield > 0 else counts
                
                bkg_hists.append(norm_counts)
                bkg_labels.append(process_labels.get(proc, proc))
                bkg_colors_list.append(mc_colors[proc])
                
            # Plot Background Stack
            if bkg_hists:
                hep.histplot(bkg_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                             label=bkg_labels, color=bkg_colors_list, sort='yield')
                             
            # 2. Gather & Normalize Signals (Overlaid, Unstacked)
            for proc in sig_processes:
                counts = hist_dictionary[var_name][proc]['counts']
                sig_yield = np.sum(counts)
                
                # Normalize each signal to 1 independently to compare shape
                norm_counts = counts / sig_yield if sig_yield > 0 else counts
                
                hep.histplot(norm_counts, bins=bins, ax=ax, stack=False, histtype='step', 
                             linewidth=3, label=f"{process_labels.get(proc, proc)} (Sig)", color=mc_colors[proc])

            # Styling
            ax.set_xlabel(var_name)
            ax.set_ylabel("Arbitrary Units (Normalized to 1)")
            ax.legend(loc='upper right', ncol=2, fontsize=12) 
            ax.set_ylim(0, ax.get_ylim()[1] * 1.3) # Extra headroom for legend
            ax.set_xlim(0,0.8)
            #ax.set_yscale('log')
            
            hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.6, fontsize=14)

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig) 
        
    print(f"\nFinished generating plots. Saved to {output_filename}")

# ==========================================
# 5. EXECUTE
# ==========================================
plot_shapes_to_pdf(tagger_vars, hist_dict, "Tagger_Shape_Comparison.pdf")


--- Starting Data Extraction & Histogramming ---
Extracting nominal data from 4 files...

Finished generating plots. Saved to Tagger_Shape_Comparison.pdf


In [32]:
# This is the data vs MC:

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

def cuts(df_):
    base = (
        (df_['n_ak4jets']   >= 5)       &
        (df_['n_b_outZH']   == 2)       &
        (df_['ZH_M']        >= 50)      &
        (df_['MET_pt']      > 20)       &
        (df_['ZH_pt'] >= 200) &
        (df_['ZH_M']        <= 200)
    )
    return base

# ==========================================
# 1. GLOBAL SETTINGS & CONFIGURATION
# ==========================================
hep.style.use(hep.style.CMS)

COFFEA_DIR = "Taggers/" # <-- UPDATE THIS PATH
LUMI = 110 # in fb^-1 

# --- Processes (Data included) ---
bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
sig_processes = ['ttZ', 'ttH']
data_process  = 'data_obs'
all_processes = bkg_processes + sig_processes + [data_process]
mc_processes  = bkg_processes + sig_processes

# --- Variables to Compare ---
tagger_vars = [
    #'ZH_GloParT3_Xbb', 
    #'ZH_GloParT3_QCD', 
    'ZH_PNet_XbbVsQCD', 
    'ZH_PNetLegacy_XbbVsQCD',
    'ZH_bbvLscore', 
]
cut_vars = ['ZH_M', 'n_b_outZH', 'n_ak4jets', 'MET_pt', 'ZH_pt']

binning_dict = {
    'ZH_GloParT3_Xbb': (40, 0, 1),
    'ZH_GloParT3_QCD': (40, 0, 1),
    'ZH_PNet_XbbVsQCD': (40, 0, 1),
    'ZH_PNetLegacy_XbbVsQCD': (40, 0, 1),
    'ZH_bbvLscore': (40, 0, 1)
}
default_binning = (40, 0, 1)

bkg_colors = {'VJets':'#3f90da', 'QCD':'#ffa90e', 'tt_B':'#bd1f01', 'TTBar':'#94a4a2', 'SingleTop':'#e76300', 'TTX':'#b9ac70'}
sig_colors = {'ttZ':'#832db6', 'ttH':'#a96b59'}
mc_colors = {**bkg_colors, **sig_colors}

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf, t\bar{t}+cc$', 'SingleTop': 'single top',
    'TTX': r'$t\bar{t}t\bar{t}, t\bar{t}W, t\bar{t}HW, t\bar{t}Hq$',
    'ttZ': r'$t\bar{t}Z$', 'ttH': r'$t\bar{t}H$', 'data_obs': 'Data', 'QCD': 'QCD'
}

# ==========================================
# 2. DATA EXTRACTION & WEIGHTING FUNCTIONS
# ==========================================
def extract_nominal_genweights(coffea_dir):
    target_weights = {
        'tttolnu2q': 480547550.0, 'ttto2l2nu': 466318140.0, 'ttto4q': 468705400.0,
        'ttbbtolnu2q': 21007254.0, 'ttbbto2l2nu': 11683236.0, 'ttbbto4q': 14012432.0
    }
    nom_genweights = {}
    nom_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    
    for file_path in nom_files:
        filein = load(file_path)
        gw_dict = filein.get('sum_signOf_genweights', {})
        for dataset, weight in gw_dict.items():
            if isinstance(weight, dict):
                for sub_dataset, sub_weight in weight.items():
                    clean_name = sub_dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                    nom_genweights[sub_dataset] = target_weights.get(clean_name, sub_weight)
            else:
                clean_name = dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                nom_genweights[dataset] = target_weights.get(clean_name, weight)
    return nom_genweights

def getZhbbWeight(df_, year=None):
    if 'norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) 

    weight = df_['norm_weight'].copy()
    
    # ==========================================
    # THE LUMINOSITY PATCH
    # Rescale from 2017 default to 2024 target
    # ==========================================
    old_lumi = 41.529
    target_lumi = 110.0
    weight *= (target_lumi / old_lumi)
    # ==========================================

    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    
    # Be sure to uncomment 'btag_sf' now that the normalization is fixed!
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf','topptWeight', 'puWeight', 'ele_trig_sf', 'btag_sf']  
    
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
        
    return weight

def load_and_cut_data(coffea_dir=COFFEA_DIR, nom_genweights=None):
    tracked_data = {proc: {} for proc in all_processes}
    valid_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    
    print(f"Extracting nominal data from {len(valid_files)} files...")

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            if 'TTTo' in raw_proc: mapped_proc = 'TTBar' if 'tt+B' not in raw_proc else None
            elif 'TTbb' in raw_proc: mapped_proc = 'tt_B' if 'tt+B' in raw_proc else None
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: mapped_proc = 'VJets'
            elif 'QCD' in raw_proc: mapped_proc = 'QCD'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower(): mapped_proc = 'ttZ'
            elif 'TTLL' in raw_proc or 'TTNuNu' in raw_proc: mapped_proc = 'ttZ'
            elif 'SingleTop' in raw_proc: mapped_proc = 'SingleTop'
            elif 'TTX' in raw_proc: mapped_proc = 'TTX'
            elif 'DATA' in raw_proc or 'data' in raw_proc.lower(): mapped_proc = 'data_obs'
            
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():
                # Data requires no genweights or norm weights
                if mapped_proc == 'data_obs':
                    current_vars = tagger_vars + cut_vars
                    genweight = 1.0
                else:
                    current_vars = tagger_vars + ['norm_weight', 'genWeight'] + cut_vars +['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'puWeight_up', 'mu_trig_sf','topptWeight', 'btag_sf', 'ele_trig_sf']
                    genweight = nom_genweights.get(dataset, 1.0) if nom_genweights else genweight_dict.get(dataset, 1.0)
                    if isinstance(genweight, dict): genweight = genweight.get(dataset, 1.0)
                
                try:
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue 
                    
                tmp_data = {}
                skip_dataset = False
                
                for var in current_vars:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    
                    if dict_key in nom_dict:
                        arr = np.array(nom_dict[dict_key].value)
                        if var == 'norm_weight':
                            tmp_data[var] = arr / genweight
                        else:
                            tmp_data[var] = arr
                    elif var in cut_vars or var == 'norm_weight':
                        skip_dataset = True
                        break 
                        
                if skip_dataset or not tmp_data: continue 
                
                df = pd.DataFrame(tmp_data)
                if dataset not in tracked_data[mapped_proc]:
                    tracked_data[mapped_proc][dataset] = []
                tracked_data[mapped_proc][dataset].append(df)

    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = []
        for dataset, branches in tracked_data[proc].items():
            dfs_to_concat.extend(branches)
            
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            df = df[cuts(df)] 
            df['tot_weight'] = getZhbbWeight(df) if proc != 'data_obs' else 1.0
            data_dict[proc] = df
        else:
            data_dict[proc] = None
            
    return data_dict

# ==========================================
# 3. PRE-COMPUTE HISTOGRAMS & BLINDING
# ==========================================
print("\n--- Starting Data Extraction & Histogramming ---")
global_nom_genweights = extract_nominal_genweights(COFFEA_DIR)
current_data = load_and_cut_data(nom_genweights=global_nom_genweights)

hist_dict = {v: {} for v in tagger_vars}

print(len(current_data['data_obs']))

for var in tagger_vars:
    n_bins, x_min, x_max = binning_dict.get(var, default_binning)
    bins = np.linspace(x_min, x_max, n_bins + 1)
    
    for proc in all_processes:
        df = current_data[proc]
        if df is None or df.empty or var not in df.columns:
            hist_dict[var][proc] = {'counts': np.zeros(n_bins), 'err2': np.zeros(n_bins)}
            continue
            
        mask = df[var].notna()
        vals = np.clip(df[var][mask], None, bins[-1])
        weights = df['tot_weight'][mask].copy()
        
        if proc == 'tt_B': weights *= 1 #1.318 * 1.8 # k-factor
        
        counts, _ = np.histogram(vals, bins=bins, weights=weights)
        err2, _   = np.histogram(vals, bins=bins, weights=weights**2)
        
        # ---------------------------------------------------------
        # APPLY BLINDING TO DATA
        # ---------------------------------------------------------
        if proc == 'data_obs':
            counts = counts.astype(float)
            err2 = err2.astype(float)
            
            # Find bins where the left edge is >= 0.8
            blind_mask = (bins[:-1] >= 0.8)
            
            # Set blinded bins to NaN so matplotlib drops them
            counts[blind_mask] = np.nan
            err2[blind_mask] = np.nan

        hist_dict[var][proc] = {'counts': counts, 'err2': err2}

# ==========================================
# 4. PLOTTING FUNCTION (DATA VS MC)
# ==========================================
def plot_data_mc_pdf(var_names, hist_dictionary, output_filename="Tagger_Data_MC.pdf"):
    with PdfPages(output_filename) as pdf:
        
        # Create a 2x2 grid for 4 variables. Size is massive to allow high res viewing
        fig = plt.figure(figsize=(22, 22))
        outer_grid = fig.add_gridspec(3, 2, wspace=0.25, hspace=0.3)
        
        for idx, var_name in enumerate(var_names):
            row, col = idx // 2, idx % 2
            
            # Inner grid: 3 parts top for Main plot, 1 part bottom for Ratio plot
            inner_grid = outer_grid[row, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
            ax = fig.add_subplot(inner_grid[0])
            rax = fig.add_subplot(inner_grid[1], sharex=ax)
            ax.tick_params(labelbottom=False)
            
            n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
            bins = np.linspace(x_min, x_max, n_bins + 1)
            bin_centers = 0.5 * (bins[1:] + bins[:-1])
            
            mc_hists, mc_labels, mc_colors_list = [], [], []
            total_mc_counts = np.zeros(n_bins)
            total_mc_stat_err2 = np.zeros(n_bins)
            
            # 1. Stack MC Backgrounds + Signals
            for proc in mc_processes:
                nom_data = hist_dictionary[var_name][proc]
                counts, err2 = nom_data['counts'], nom_data['err2']
                yield_total, stat_unc = np.sum(counts), np.sqrt(np.sum(err2))
                
                mc_hists.append(counts)
                mc_labels.append(f"{process_labels.get(proc, proc)} ({yield_total:.1f})")
                mc_colors_list.append(mc_colors[proc])
                
                total_mc_counts += counts
                total_mc_stat_err2 += err2

            total_mc_err = np.sqrt(total_mc_stat_err2)

            if mc_hists:
                hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                             label=mc_labels, color=mc_colors_list, sort='yield')

            # Stat uncertainty hatched band
            ax.stairs(values=total_mc_counts + total_mc_err, 
                      baseline=np.clip(total_mc_counts - total_mc_err, 0, None),
                      edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none', 
                      label=r'Stat Unc.')

            # 2. Process Data (Blinded bins are handled safely as NaNs)
            data_counts = hist_dictionary[var_name]['data_obs']['counts']
            data_err = np.sqrt(hist_dictionary[var_name]['data_obs']['err2'])
            
            # Use nansum to ignore the blinded bins in the total yield calculation
            data_yield = np.nansum(data_counts)
            print(data_yield)
            print(np.sum(data_counts))
            
            if data_yield > 0:
                data_lbl = f"Data ({data_yield})"
                hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar', 
                             color='black', label=data_lbl, yerr=data_err)

            # 3. Ratio Panel
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio = data_counts / total_mc_counts
                ratio_err = np.abs(data_err / total_mc_counts) 
                mc_rel_err = np.abs(total_mc_err / total_mc_counts)

            for arr in [ratio, ratio_err, mc_rel_err]:
                # Preserve nans for data, but protect against inf from zero-MC bins
                arr[np.isinf(arr)] = 0

            yerr_down = np.clip(ratio_err, 0, np.maximum(ratio, 0))

            rax.stairs(values=1 + mc_rel_err, 
                       baseline=np.clip(1 - mc_rel_err, 0, None),
                       edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none')
                       
            rax.errorbar(bin_centers, ratio, yerr=[yerr_down, ratio_err], fmt='ko', markersize=4)
            rax.axhline(1, color='black', linestyle='--')
            
            # 4. Styling
            ax.set_ylabel("Events")
            ax.legend(loc='upper right', ncol=2, fontsize=10) 
            
            # Safeguard max_val calculation against arrays full of NaNs
            max_mc = np.max(total_mc_counts)
            max_data = np.nanmax(data_counts) if not np.isnan(data_counts).all() else 0
            max_val = max(max_mc, max_data)
            
            ax.set_ylim(0.1, max_val * 50 if max_val > 0 else 100)
            ax.set_yscale('log')
            
            rax.set_xlabel(var_name)
            rax.set_ylabel("Data / MC")
            rax.set_ylim(0, 2) # Zoomed in ratio since stat-only MC usually tight
            ax.set_xlim(0, 1)
            
            hep.cms.label("Preliminary", data=(data_yield > 0), lumi=LUMI, ax=ax, com=13.6, fontsize=14)

        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig) 
        
    print(f"\nFinished generating plots. Saved to {output_filename}")

# ==========================================
# 5. EXECUTE
# ==========================================
plot_data_mc_pdf(tagger_vars, hist_dict, "Tagger_Data_MC_Blinded.pdf")


--- Starting Data Extraction & Histogramming ---
Extracting nominal data from 4 files...
70699
65469.0
nan
64068.0
nan
69590.0
nan

Finished generating plots. Saved to Tagger_Data_MC_Blinded.pdf


In [2]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

def cuts(df_):
    base = (
        (df_['n_ak4jets']   >= 5)       &
        (df_['n_b_outZH']   == 2)       &
        #(df_['ZH_PNetLegacy_XbbVsQCD'] > 0.8) &
        (df_['ZH_M']        >= 50)      &
        (df_['MET_pt'] > 20) &
        (df_['ZH_M']        <= 200)
    )
    return base

# ==========================================
# 1. GLOBAL SETTINGS & CONFIGURATION
# ==========================================
hep.style.use(hep.style.CMS)

COFFEA_DIR = "Taggers/" # <-- UPDATE THIS PATH
LUMI = 110 # in fb^-1 

# --- Processes ---
bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
sig_processes = ['ttZ', 'ttH']
data_process = 'data_obs'
all_processes = bkg_processes + sig_processes + [data_process]

# --- Variables ---
weight_vars = [
    'genWeight', 'norm_weight', 'topptWeight', 'ele_reco_sf', 'ele_id_sf', 
    'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf', 'puWeight', 'btag_sf', 'ele_trig_sf'
]

validation_vars = [
    'nPV', 'nPVGood', 'MET_pt', 'MET_phi', 'lep_pt', 'ele_pt', 'muon_pt',
    'lep_eta', 'ele_eta', 'muon_eta', 'n_ak4', 'n_bjet', 'n_ak8',
    'jet1_pt', 'jet2_pt', 'bjet1_pt', 
    'jet1_eta', 'jet2_eta', 'bjet1_eta', 
    'jet1_btag', 'jet2_btag', 'bjet1_btag', 
    'fatjet1_pt', 'fatjet1_eta', 'fatjet1_mass', 'n_b_outZH', 'n_ak4jets', 'ZH_bbvLscore', 'ZH_PNetLegacy_XbbVsQCD'
]

# Override for quick testing (as in your original script)
validation_vars = ['MET_phi', 'lep_pt', 'lep_eta']

cut_vars = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'MET_pt', 'ZH_PNetLegacy_XbbVsQCD', 'ele_pt', 'muon_pt']
vars_to_extract = validation_vars + ['norm_weight'] + weight_vars + cut_vars

binning_dict = {
    'nPV': (50, 0, 100), 'nPVGood': (50, 0, 100),
    'MET_pt': (40, 0, 800), 'MET_phi': (30, -3.14, 3.14),
    'lep_pt': (40, 0, 800), 'ele_pt': (40, 0, 800), 'muon_pt': (40, 0, 800),
    'lep_eta': (30, -2.5, 2.5), 'ele_eta': (30, -2.5, 2.5), 'muon_eta': (30, -2.4, 2.4),
    'n_ak4jets': (15, 0, 15), 'n_bjet': (10, 0, 10), 'n_ak8': (5, 0, 5),
    'jet1_pt': (40, 0, 1000), 'jet2_pt': (40, 0, 800),
    'bjet1_pt': (40, 0, 800), 'bjet2_pt': (40, 0, 600),
    'jet1_eta': (30, -2.5, 2.5), 'jet2_eta': (30, -2.5, 2.5),
    'bjet1_eta': (30, -2.5, 2.5), 'bjet2_eta': (30, -2.5, 2.5),
    'jet1_btag': (20, 0, 1), 'jet2_btag': (20, 0, 1),
    'bjet1_btag': (20, 0, 1), 'bjet2_btag': (20, 0, 1),
    'fatjet1_pt': (40, 200, 1200), 'fatjet1_eta': (30, -2.5, 2.5), 'fatjet1_mass': (40, 0, 400),
    'ZH_bbvLscore': (30, 0, 1)
}
default_binning = (40, 0, 500)

bkg_colors = {'VJets':'#3f90da', 'QCD':'#ffa90e', 'tt_B':'#bd1f01', 'TTBar':'#94a4a2', 'SingleTop':'#e76300', 'TTX':'#b9ac70'}
sig_colors = {'ttZ':'#832db6', 'ttH':'#a96b59'}
mc_colors = {**bkg_colors, **sig_colors}

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf, t\bar{t}+cc$', 'SingleTop': 'single top',
    'TTX': r'$t\bar{t}t\bar{t}, t\bar{t}W, t\bar{t}HW, t\bar{t}Hq$',
    'ttZ': r'$t\bar{t}Z$', 'ttH': r'$t\bar{t}H$', 'data_obs': 'Data'
}

# ==========================================
# 2. DATA EXTRACTION & WEIGHTING FUNCTIONS
# ==========================================
def extract_nominal_genweights(coffea_dir):
    target_weights = {
        'tttolnu2q': 480547550.0,
        'ttto2l2nu': 466318140.0,
        'ttto4q': 468705400.0,
        'ttbbtolnu2q': 21007254.0,
        'ttbbto2l2nu': 11683236.0,
        'ttbbto4q': 14012432.0
    }
    
    nom_genweights = {}
    nom_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    
    for file_path in nom_files:
        filein = load(file_path)
        gw_dict = filein.get('sum_signOf_genweights', {})
        
        for dataset, weight in gw_dict.items():
            if isinstance(weight, dict):
                for sub_dataset, sub_weight in weight.items():
                    clean_name = sub_dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                    if clean_name in target_weights:
                        nom_genweights[sub_dataset] = target_weights[clean_name]
                    else:
                        nom_genweights[sub_dataset] = sub_weight
            else:
                clean_name = dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                if clean_name in target_weights:
                    nom_genweights[dataset] = target_weights[clean_name]
                else:
                    nom_genweights[dataset] = weight
                    
    print("\n--- Final Forced Genweights ---")
    for k, v in nom_genweights.items():
        if 'tt' in k.lower():
            print(f"{k}: {v}")
            
    return nom_genweights

def getZhbbWeight(df_, year=None):
    if 'norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) 

    weight = df_['norm_weight'].copy()
    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    #sfs = []
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'puWeight', 'mu_trig_sf','topptWeight', 'btag_sf', 'ele_trig_sf']
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
        
    return weight

def load_and_cut_data(coffea_dir=COFFEA_DIR, year=2024, nom_genweights=None):
    tracked_data = {proc: {} for proc in all_processes}
    valid_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    
    if len(valid_files) == 0:
        print("  -> [WARNING] No nominal files found!")
        return {}
        
    print(f"Extracting nominal distributions from {len(valid_files)} matching files...")

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            if 'TTTo' in raw_proc:
                if 'tt+B' in raw_proc: continue 
                mapped_proc = 'TTBar'
            elif 'TTbb' in raw_proc:
                if 'tt+B' not in raw_proc: continue 
                mapped_proc = 'tt_B'
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: mapped_proc = 'VJets'
            elif 'QCD' in raw_proc: mapped_proc = 'QCD'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower(): mapped_proc = 'ttZ'
            elif 'TTLL' in raw_proc or 'TTNuNu' in raw_proc: mapped_proc = 'ttZ'
            elif 'DATA' in raw_proc or 'data' in raw_proc.lower(): mapped_proc = 'data_obs'
            elif 'SingleTop' in raw_proc: mapped_proc = 'SingleTop'
            elif 'TTX' in raw_proc: mapped_proc = 'TTX'
            
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():
                if nom_genweights is not None:
                    genweight = nom_genweights.get(dataset, 1.0)
                    print(mapped_proc, dataset, genweight)
                else:
                    genweight = genweight_dict.get(dataset, 1.0) 
                    if isinstance(genweight, dict):
                        genweight = genweight.get(dataset, 1.0)
                        
                if genweight == 1.0 and mapped_proc != 'data_obs':
                    print(f"  -> [WARNING] Nominal genweight missing for {dataset}! Defaulting to 1.0.")
                
                try:
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue 
                    
                if mapped_proc == 'data_obs':
                    current_vars_to_extract = validation_vars + cut_vars
                    essential_vars = ['n_b_outZH', 'n_ak4jets'] + cut_vars
                else:
                    current_vars_to_extract = vars_to_extract + cut_vars
                    essential_vars = ['n_b_outZH', 'n_ak4jets', 'norm_weight'] + cut_vars
                
                tmp_data = {}
                skip_dataset = False
                
                for var in current_vars_to_extract:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    
                    if dict_key in nom_dict:
                        arr = np.array(nom_dict[dict_key].value)
                    elif var in essential_vars:
                        print(f"🚨 FATAL: Essential '{dict_key}' missing in {mapped_proc} ({dataset})!")
                        skip_dataset = True
                        break 
                    else:
                        continue 
                        
                    if var == 'norm_weight' and mapped_proc != 'data_obs':
                        tmp_data[var] = arr / genweight
                    else:
                        tmp_data[var] = arr
                        
                if skip_dataset: continue 
                
                if tmp_data: 
                    df = pd.DataFrame(tmp_data)
                    if dataset not in tracked_data[mapped_proc]:
                        tracked_data[mapped_proc][dataset] = []
                    tracked_data[mapped_proc][dataset].append(df)


    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = []
        for dataset, branches in tracked_data[proc].items():
            dfs_to_concat.extend(branches)
            
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            df = df[cuts(df)] 
            df['tot_weight'] = getZhbbWeight(df, year=year) if proc != 'data_obs' else 1.0
            data_dict[proc] = df
        else:
            data_dict[proc] = None
            
    return data_dict

# ==========================================
# 3. PRE-COMPUTE HISTOGRAMS TO SAVE RAM
# ==========================================
print("\n--- Starting Data Extraction & Histogramming ---")

print("Extracting true nominal genweights...")
global_nom_genweights = extract_nominal_genweights(COFFEA_DIR)

hist_dict = {v: {p: {} for p in all_processes} for v in validation_vars}

current_data = load_and_cut_data(nom_genweights=global_nom_genweights)

for var in validation_vars:
    n_bins, x_min, x_max = binning_dict.get(var, default_binning)
    bins = np.linspace(x_min, x_max, n_bins + 1)
    
    for proc in all_processes:
        df = current_data.get(proc)
        if df is None: continue
            
        if df.empty or var not in df.columns:
            hist_dict[var][proc]['nominal'] = {'counts': np.zeros(n_bins), 'err2': np.zeros(n_bins)}
            continue
            
        mask = df[var].notna()
        vals = np.clip(df[var][mask], None, bins[-1])
        weights = df['tot_weight'][mask].copy()
        
        if proc == 'tt_B': weights *= 5.4 #1.318 * 1.8 # k-factor
        
        # Compute Nominal & Sum-of-Weights-Squared (Stat Uncertainty)
        counts, _ = np.histogram(vals, bins=bins, weights=weights)
        err2, _ = np.histogram(vals, bins=bins, weights=weights**2)
        hist_dict[var][proc]['nominal'] = {'counts': counts, 'err2': err2}

del current_data

# ==========================================
# 4. PLOTTING FUNCTION
# ==========================================
def plot_variables_to_pdf(var_names, hist_dictionary, output_filename="Data_MC_Nominal_StatOnly.pdf"):
    mc_processes = bkg_processes + sig_processes
    
    with PdfPages(output_filename) as pdf:
        for chunk_start in range(0, len(var_names), 9):
            chunk_vars = var_names[chunk_start : chunk_start + 9]
            
            fig = plt.figure(figsize=(24, 24))
            outer_grid = fig.add_gridspec(3, 3, wspace=0.3, hspace=0.3)
            
            for idx, var_name in enumerate(chunk_vars):
                row, col = idx // 3, idx % 3
                inner_grid = outer_grid[row, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.00)
                ax = fig.add_subplot(inner_grid[0])
                rax = fig.add_subplot(inner_grid[1], sharex=ax)
                ax.tick_params(labelbottom=False)
                
                n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
                bins = np.linspace(x_min, x_max, n_bins + 1)
                bin_centers = 0.5 * (bins[1:] + bins[:-1])
                
                mc_hists, mc_labels, mc_colors_list = [], [], []
                total_mc_counts = np.zeros(n_bins)
                total_mc_stat_err2 = np.zeros(n_bins)
                
                # Stack Nominal
                for proc in mc_processes:
                    nom_data = hist_dictionary[var_name][proc].get('nominal')
                    if not nom_data: continue
                        
                    counts, err2 = nom_data['counts'], nom_data['err2']
                    yield_total, stat_unc = np.sum(counts), np.sqrt(np.sum(err2))
                    
                    mc_hists.append(counts)
                    mc_labels.append(f"{process_labels.get(proc, proc)} ({yield_total:.1f} ± {stat_unc:.1f})")
                    mc_colors_list.append(mc_colors[proc])
                    
                    total_mc_counts += counts
                    total_mc_stat_err2 += err2

                # Total Uncertainty = Stat Only
                total_mc_err = np.sqrt(total_mc_stat_err2)

                if mc_hists:
                    hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                                 label=mc_labels, color=mc_colors_list, sort='yield')

                ax.stairs(values=total_mc_counts + total_mc_err, 
                    baseline=np.clip(total_mc_counts - total_mc_err, 0, None),
                    edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none', 
                    label='Stat Unc.')

                # --- Process Data ---
                data_dict_entry = hist_dictionary[var_name].get(data_process, {}).get('nominal')
                if data_dict_entry:
                    data_counts = data_dict_entry['counts']
                    data_err = np.sqrt(data_counts)
                    data_yield = np.sum(data_counts)
                    
                    if data_yield > 0:
                        data_lbl = f"Data ({data_yield:.0f} ± {np.sqrt(data_yield):.1f})"
                        hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar', 
                                     color='black', label=data_lbl, yerr=data_err)

                    # --- Ratio Panel ---
                    with np.errstate(divide='ignore', invalid='ignore'):
                        ratio = data_counts / total_mc_counts
                        ratio_err = np.abs(data_err / total_mc_counts) 
                        mc_rel_err = np.abs(total_mc_err / total_mc_counts)

                    for arr in [ratio, ratio_err, mc_rel_err]:
                        arr[np.isnan(arr) | np.isinf(arr)] = 0

                    yerr_down = np.clip(ratio_err, 0, np.maximum(ratio, 0))

                    rax.stairs(values=1 + mc_rel_err, 
                               baseline=np.clip(1 - mc_rel_err, 0, None),
                               edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none')
                               
                    rax.errorbar(bin_centers, ratio, yerr=[yerr_down, ratio_err], fmt='ko', markersize=3)
                    
                rax.axhline(1, color='black', linestyle='--')
                
                # --- Styling ---
                ax.set_ylabel("Events")
                ax.legend(loc='upper right', ncol=2, fontsize=10) 
                
                if data_dict_entry:
                    max_val = max(np.max(total_mc_counts), np.max(data_counts))
                else:
                    max_val = np.max(total_mc_counts)
                    
                ax.set_ylim(0.1, max_val * 100 if max_val > 0 else 100)
                ax.set_yscale('log')
                
                rax.set_xlabel(var_name)
                rax.set_ylabel("Data / MC")
                rax.set_ylim(0, 2)
                
                hep.cms.label("Preliminary", data=bool(data_dict_entry and data_yield > 0), lumi=LUMI, ax=ax, com=13.6, fontsize=12)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig) 
            
    print(f"\nFinished generating plots. Saved to {output_filename}")

# ==========================================
# 5. EXECUTE
# ==========================================
plot_variables_to_pdf(validation_vars, hist_dict, "Data_MC_Nominal_StatOnly.pdf")


--- Starting Data Extraction & Histogramming ---
Extracting true nominal genweights...

--- Final Forced Genweights ---
TTZ-ZtoQQ-1Jets_2024: 1595505.0
TTH-HtoNon2B_2024: 52796792.0
TTNuNu_2024: 2075742.0
TTLL_Bin-MLL-50_2024: 2054714.0
TTLL_Bin-MLL-4to50_2024: 1645190.0
TTH-Hto2B_2024: 2390882.0
TTtoLNu2Q_2024: 480547550.0
TTto2L2Nu_2024: 466318140.0
TTto4Q_2024: 468705400.0
TTBBtoLNu2Q_2024: 21007254.0
TTBBto2L2Nu_2024: 11683236.0
TTBBto4Q_2024: 14012432.0
Extracting nominal distributions from 3 matching files...
ttZ TTZ-ZtoQQ-1Jets_2024 1595505.0
ttZ TTZ-ZtoQQ-1Jets_2024 1595505.0
ttH TTH-HtoNon2B_2024 52796790.0
ttH TTH-HtoNon2B_2024 52796790.0
ttZ TTNuNu_2024 2075742.0
ttZ TTLL_Bin-MLL-50_2024 2054714.0
ttZ TTLL_Bin-MLL-4to50_2024 1645190.0
ttZ TTLL_Bin-MLL-50_2024 2054714.0
ttZ TTLL_Bin-MLL-4to50_2024 1645190.0
ttH TTH-Hto2B_2024 2390882.0
ttH TTH-Hto2B_2024 2390882.0
data_obs DATA_Muon_2024_EraI 1.0
data_obs DATA_Muon_2024_EraH 1.0
data_obs DATA_Muon_2024_EraG 1.0
data_obs DATA

In [7]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

def cuts(df_):
    base = (
        (df_['nPVGood']     >= 1)       & # Baseline collision filter
        (df_['n_ak4jets']   >= 5)       &
        (df_['n_b_outZH']   >= 1)       & # Relaxed to allow both 3-tag and 4-tag regions
        #(df_['ZH_bbvLscore'] < 0.5669)  &
        (df_['ZH_M']        >= 50)      &
        (df_['MET_pt']      > 20)       &
        (df_['ZH_M']        <= 200)
    )
    return base

# ==========================================
# 1. GLOBAL SETTINGS & CONFIGURATION
# ==========================================
hep.style.use(hep.style.CMS)

COFFEA_DIR = "DataVsMC/" # <-- UPDATE THIS PATH
LUMI = 110 # in fb^-1 

bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
sig_processes = ['ttZ', 'ttH']
data_process = 'data_obs'
all_processes = bkg_processes + sig_processes + [data_process]
mc_processes = bkg_processes + sig_processes

weight_vars = [
    'genWeight', 'norm_weight', 'topptWeight', 'ele_reco_sf', 'ele_id_sf', 
    'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf', 'puWeight', 'btag_sf'
]

validation_vars = ['MET_phi', 'lep_pt', 'lep_eta', 'ZH_bbvLscore']
cut_vars = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'MET_pt', 'nPVGood']
vars_to_extract = validation_vars + ['norm_weight'] + weight_vars + cut_vars

binning_dict = {
    'MET_phi': (30, -3.14, 3.14),
    'lep_pt': (40, 0, 800),
    'lep_eta': (30, -2.5, 2.5),
    'ZH_bbvLscore': (40, 0, 1)
}
default_binning = (40, 0, 500)

bkg_colors = {'VJets':'#3f90da', 'QCD':'#ffa90e', 'tt_B':'#bd1f01', 'TTBar':'#94a4a2', 'SingleTop':'#e76300', 'TTX':'#b9ac70'}
sig_colors = {'ttZ':'#832db6', 'ttH':'#a96b59'}
mc_colors = {**bkg_colors, **sig_colors}

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf, t\bar{t}+cc$', 'SingleTop': 'single top',
    'TTX': r'$t\bar{t}t\bar{t}, t\bar{t}W, t\bar{t}HW, t\bar{t}Hq$',
    'ttZ': r'$t\bar{t}Z$', 'ttH': r'$t\bar{t}H$', 'data_obs': 'Data'
}

# ==========================================
# 2. DATA EXTRACTION
# ==========================================
def extract_nominal_genweights(coffea_dir):
    target_weights = {
        'tttolnu2q': 480547550.0, 'ttto2l2nu': 466318140.0, 'ttto4q': 468705400.0,
        'ttbbtolnu2q': 21007254.0, 'ttbbto2l2nu': 11683236.0, 'ttbbto4q': 14012432.0
    }
    nom_genweights = {}
    for file_path in [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]:
        filein = load(file_path)
        gw_dict = filein.get('sum_signOf_genweights', {})
        for dataset, weight in gw_dict.items():
            if isinstance(weight, dict):
                for sub_dataset, sub_weight in weight.items():
                    clean_name = sub_dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                    nom_genweights[sub_dataset] = target_weights.get(clean_name, sub_weight)
            else:
                clean_name = dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                nom_genweights[dataset] = target_weights.get(clean_name, weight)
    return nom_genweights

def getZhbbWeight(df_, year=None):
    if 'norm_weight' not in df_.columns: return pd.Series(1.0, index=df_.index) 
    weight = df_['norm_weight'].copy()
    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    for sf in ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'puWeight', 'mu_trig_sf','topptWeight', 'btag_sf']:
        weight *= df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
    return weight

def load_and_cut_data(coffea_dir=COFFEA_DIR, year=2024, nom_genweights=None):
    tracked_data = {proc: {} for proc in all_processes}
    valid_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    
    print(f"Extracting nominal distributions from {len(valid_files)} matching files...")
    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            if 'TTTo' in raw_proc: mapped_proc = 'TTBar' if 'tt+B' not in raw_proc else None
            elif 'TTbb' in raw_proc: mapped_proc = 'tt_B' if 'tt+B' in raw_proc else None
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: mapped_proc = 'VJets'
            elif 'QCD' in raw_proc: mapped_proc = 'QCD'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower(): mapped_proc = 'ttZ'
            elif 'TTLL' in raw_proc or 'TTNuNu' in raw_proc: mapped_proc = 'ttZ'
            elif 'DATA' in raw_proc or 'data' in raw_proc.lower(): mapped_proc = 'data_obs'
            elif 'SingleTop' in raw_proc: mapped_proc = 'SingleTop'
            elif 'TTX' in raw_proc: mapped_proc = 'TTX'
            
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():
                genweight = nom_genweights.get(dataset, 1.0) if nom_genweights else genweight_dict.get(dataset, 1.0)
                if isinstance(genweight, dict): genweight = genweight.get(dataset, 1.0)
                
                try: nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError: continue 
                    
                current_vars = (validation_vars + cut_vars) if mapped_proc == 'data_obs' else (vars_to_extract + cut_vars)
                essential_vars = ['n_b_outZH', 'n_ak4jets'] + cut_vars if mapped_proc == 'data_obs' else ['n_b_outZH', 'n_ak4jets', 'norm_weight'] + cut_vars
                
                tmp_data = {}
                skip_dataset = False
                
                for var in current_vars:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    if dict_key in nom_dict:
                        arr = np.array(nom_dict[dict_key].value)
                        tmp_data[var] = (arr / genweight) if (var == 'norm_weight' and mapped_proc != 'data_obs') else arr
                    elif var in essential_vars:
                        skip_dataset = True
                        break 
                        
                if skip_dataset or not tmp_data: continue 
                
                df = pd.DataFrame(tmp_data)
                if dataset not in tracked_data[mapped_proc]: tracked_data[mapped_proc][dataset] = []
                tracked_data[mapped_proc][dataset].append(df)

    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = []
        for dataset, branches in tracked_data[proc].items(): dfs_to_concat.extend(branches)
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            df = df[cuts(df)] 
            df['tot_weight'] = getZhbbWeight(df, year=year) if proc != 'data_obs' else 1.0
            data_dict[proc] = df
        else: data_dict[proc] = None
    return data_dict

# ==========================================
# 3. CALCULATE SCALE FACTOR & HISTOGRAM
# ==========================================
global_nom_genweights = extract_nominal_genweights(COFFEA_DIR)
current_data = load_and_cut_data(nom_genweights=global_nom_genweights)

print("\n" + "="*50)
print(" DERIVING tt+bb SCALE FACTOR IN 3-TAG CONTROL REGION")
print("="*50)

data_yield_3tag = 0
if current_data.get('data_obs') is not None:
    mask_3tag = current_data['data_obs']['n_b_outZH'] == 1
    data_yield_3tag = np.sum(current_data['data_obs']['tot_weight'][mask_3tag])

light_mc_yield_3tag = 0
ttb_raw_yield_3tag = 0

for proc in mc_processes:
    df = current_data.get(proc)
    if df is None or df.empty: continue
    
    mask_3tag = df['n_b_outZH'] == 1
    yield_proc = np.sum(df['tot_weight'][mask_3tag])
    
    if proc == 'tt_B':
        ttb_raw_yield_3tag = yield_proc * 1.318 # Include base k-factor
    else:
        light_mc_yield_3tag += yield_proc

extra_ttb_sf = 1.0
if ttb_raw_yield_3tag > 0:
    extra_ttb_sf = (data_yield_3tag - light_mc_yield_3tag) / ttb_raw_yield_3tag

final_ttb_kfactor = 1.318 * extra_ttb_sf

print(f" Data Yield (3-tag)       : {data_yield_3tag:,.1f}")
print(f" Light MC Yield (3-tag)   : {light_mc_yield_3tag:,.1f}")
print(f" Raw tt+bb Yield (3-tag)  : {ttb_raw_yield_3tag:,.1f}")
print(f" -> Derived Extra SF      : {extra_ttb_sf:.3f}")
print(f" -> FINAL tt+bb K-FACTOR  : {final_ttb_kfactor:.3f}")
print("="*50 + "\n")

# --- Histogramming ---
regions = {'CR_3tag': 1, 'SR_4tag': 2}
hist_dict = {reg: {v: {p: {} for p in all_processes} for v in validation_vars} for reg in regions}

for reg_name, n_b_req in regions.items():
    for var in validation_vars:
        n_bins, x_min, x_max = binning_dict.get(var, default_binning)
        bins = np.linspace(x_min, x_max, n_bins + 1)
        
        for proc in all_processes:
            df = current_data.get(proc)
            if df is None or df.empty or var not in df.columns:
                hist_dict[reg_name][var][proc]['nominal'] = {'counts': np.zeros(n_bins), 'err2': np.zeros(n_bins)}
                continue
                
            reg_mask = (df['n_b_outZH'] == n_b_req)
            valid_mask = df[var].notna()
            mask = reg_mask & valid_mask
            
            vals = np.clip(df[var][mask], None, bins[-1])
            weights = df['tot_weight'][mask].copy()
            
            if proc == 'tt_B': weights *= final_ttb_kfactor 
                
            counts, _ = np.histogram(vals, bins=bins, weights=weights)
            err2, _ = np.histogram(vals, bins=bins, weights=weights**2)
            hist_dict[reg_name][var][proc]['nominal'] = {'counts': counts, 'err2': err2}

del current_data

# ==========================================
# 4. PLOTTING FUNCTION
# ==========================================
def plot_variables_to_pdf(var_names, hist_dictionary, output_filename="Data_MC_Nominal_StatOnly.pdf"):
    with PdfPages(output_filename) as pdf:
        for chunk_start in range(0, len(var_names), 9):
            chunk_vars = var_names[chunk_start : chunk_start + 9]
            
            fig = plt.figure(figsize=(24, 24))
            outer_grid = fig.add_gridspec(3, 3, wspace=0.3, hspace=0.3)
            
            for idx, var_name in enumerate(chunk_vars):
                row, col = idx // 3, idx % 3
                inner_grid = outer_grid[row, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.00)
                ax = fig.add_subplot(inner_grid[0])
                rax = fig.add_subplot(inner_grid[1], sharex=ax)
                ax.tick_params(labelbottom=False)
                
                n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
                bins = np.linspace(x_min, x_max, n_bins + 1)
                bin_centers = 0.5 * (bins[1:] + bins[:-1])
                
                mc_hists, mc_labels, mc_colors_list = [], [], []
                total_mc_counts = np.zeros(n_bins)
                total_mc_stat_err2 = np.zeros(n_bins)
                
                for proc in mc_processes:
                    nom_data = hist_dictionary[var_name][proc].get('nominal')
                    if not nom_data: continue
                    counts, err2 = nom_data['counts'], nom_data['err2']
                    yield_total, stat_unc = np.sum(counts), np.sqrt(np.sum(err2))
                    
                    mc_hists.append(counts)
                    mc_labels.append(f"{process_labels.get(proc, proc)} ({yield_total:.1f} ± {stat_unc:.1f})")
                    mc_colors_list.append(mc_colors[proc])
                    
                    total_mc_counts += counts
                    total_mc_stat_err2 += err2

                total_mc_err = np.sqrt(total_mc_stat_err2)
                if mc_hists:
                    hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                                 label=mc_labels, color=mc_colors_list, sort='yield')

                ax.stairs(values=total_mc_counts + total_mc_err, 
                    baseline=np.clip(total_mc_counts - total_mc_err, 0, None),
                    edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none', 
                    label='Stat Unc.')

                data_dict_entry = hist_dictionary[var_name].get(data_process, {}).get('nominal')
                if data_dict_entry:
                    data_counts = data_dict_entry['counts']
                    data_err = np.sqrt(data_counts)
                    data_yield = np.sum(data_counts)
                    
                    if data_yield > 0:
                        data_lbl = f"Data ({data_yield:.0f} ± {np.sqrt(data_yield):.1f})"
                        hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar', 
                                     color='black', label=data_lbl, yerr=data_err)

                    with np.errstate(divide='ignore', invalid='ignore'):
                        ratio = data_counts / total_mc_counts
                        ratio_err = np.abs(data_err / total_mc_counts) 
                        mc_rel_err = np.abs(total_mc_err / total_mc_counts)

                    for arr in [ratio, ratio_err, mc_rel_err]:
                        arr[np.isnan(arr) | np.isinf(arr)] = 0

                    yerr_down = np.clip(ratio_err, 0, np.maximum(ratio, 0))
                    rax.stairs(values=1 + mc_rel_err, baseline=np.clip(1 - mc_rel_err, 0, None),
                               edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none')
                    rax.errorbar(bin_centers, ratio, yerr=[yerr_down, ratio_err], fmt='ko', markersize=3)
                    
                rax.axhline(1, color='black', linestyle='--')
                ax.set_ylabel("Events")
                ax.legend(loc='upper right', ncol=2, fontsize=10) 
                
                max_val = max(np.max(total_mc_counts), np.max(data_counts)) if data_dict_entry else np.max(total_mc_counts)
                ax.set_ylim(0.1, max_val * 100 if max_val > 0 else 100)
                ax.set_yscale('log')
                
                rax.set_xlabel(var_name)
                rax.set_ylabel("Data / MC")
                rax.set_ylim(0, 2)
                
                hep.cms.label("Preliminary", data=bool(data_dict_entry and data_yield > 0), lumi=LUMI, ax=ax, com=13.6, fontsize=12)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig) 
    print(f"Saved: {output_filename}")

# ==========================================
# 5. EXECUTE
# ==========================================
print("\n--- Generating Plots ---")
for reg_name in regions.keys():
    plot_variables_to_pdf(validation_vars, hist_dict[reg_name], f"Data_MC_{reg_name}.pdf")

Extracting nominal distributions from 4 matching files...

 DERIVING tt+bb SCALE FACTOR IN 3-TAG CONTROL REGION
 Data Yield (3-tag)       : 76,456.0
 Light MC Yield (3-tag)   : 56,981.6
 Raw tt+bb Yield (3-tag)  : 4,303.0
 -> Derived Extra SF      : 4.526
 -> FINAL tt+bb K-FACTOR  : 5.965


--- Generating Plots ---
Saved: Data_MC_CR_3tag.pdf
Saved: Data_MC_SR_4tag.pdf
